# Homework Sesi 2 — Analisis Order E-commerce + Kalkulator Revenue

## Skenario
Anda pegang data 200 order dari toko e-commerce (`orders_hw.csv`).
Tugas Anda: jawab pertanyaan bisnis + bangun beberapa function kalkulator yang bisa
dipakai ulang.

## Kolom di `orders_hw.csv`
- `order_id` — ID unik order (string)
- `customer` — nama customer
- `product` — nama produk
- `category` — kategori: Elektronik / Fashion / Makanan / Kesehatan
- `price` — harga satuan (IDR)
- `quantity` — jumlah dibeli
- `order_date` — tanggal order (ISO)
- `city` — kota pengiriman

Perkiraan waktu: 30–45 menit.

## Cara pakai
Isi bagian `# TODO:` di tiap cell. Jika stuck, silahkan membuka file `hw2_mentor.ipynb`
sebagai referensi, namun coba dulu kerjakan semuanya sendiri.


## Setup


In [3]:
import pandas as pd

# TODO: load orders_hw.csv ke DataFrame df
df = pd.read_csv("orders_hw.csv")
print("Shape:", df.shape)
df.head()


Shape: (200, 8)


,order_id,customer,product,category,price,quantity,order_date,city
0,ORD-0001,Citra Pratama,Suplemen,Kesehatan,176611,5,2024-07-22,Makassar
1,ORD-0002,Indah Utami,Snack,Makanan,93381,8,2024-08-30,Makassar
2,ORD-0003,Eko Santoso,Sepatu,Fashion,347289,2,2024-12-05,Medan
3,ORD-0004,Lestari Permana,Suplemen,Kesehatan,63894,4,2024-09-14,Jakarta
4,ORD-0005,Kartika Lestari,Biskuit,Makanan,71064,2,2024-12-03,Surabaya


---
## Task 1 — Inspeksi Awal
Sebelum analisis apa pun, kenali dulu datanya.


In [4]:
# TODO: cetak info tipe kolom + missing values dengan df.info()
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   order_id    200 non-null    str  
 1   customer    200 non-null    str  
 2   product     200 non-null    str  
 3   category    200 non-null    str  
 4   price       200 non-null    int64
 5   quantity    200 non-null    int64
 6   order_date  200 non-null    str  
 7   city        200 non-null    str  
dtypes: int64(2), str(6)
memory usage: 23.1 KB


In [5]:
# TODO: cetak statistik ringkas kolom numerik dengan df.describe()
df.describe()

,price,quantity
count,2.000000e+02,200.000000
mean,7.262226e+05,4.515000
std,1.292504e+06,2.171023
min,1.091400e+04,1.000000
25%,6.684275e+04,3.000000
50%,1.551045e+05,5.000000
75%,4.315802e+05,6.000000
max,4.997973e+06,8.000000


---
## Task 2 — Berapa Order per Kota?


In [6]:
# TODO: pakai .value_counts() pada kolom "city"
df["city"].value_counts()


city
Makassar      33
Surabaya      31
Jakarta       27
Bandung       27
Medan         25
Yogyakarta    22
Semarang      19
Denpasar      16
Name: count, dtype: int64

---
## Task 3 — Total Revenue per Kategori
Revenue = price × quantity. Jumlahkan per kategori, urutkan dari revenue yang paling tinggi.


In [15]:
# TODO: bikin kolom "revenue_simple" = df["price"] * df["quantity"]
revenue_simple = df["price"] * df["quantity"]
df["revenue_simple"] = revenue_simple

# TODO: group by category, jumlahkan revenue_simple, sort descending
df.groupby("category")["revenue_simple"].sum().sort_values(ascending=False)

category
Elektronik    525956134
Fashion        56853501
Kesehatan      29336921
Makanan        16866934
Name: revenue_simple, dtype: int64

---
## Task 4 — Buat function untuk menghitung revenue `hitung_revenue`



In [17]:
# TODO: definisikan function hitung_revenue(price, qty, diskon=0)
def hitung_revenue(price, qty, diskon=0):
    revenue = price * qty
    revenue_setelah_diskon = revenue * (1 - diskon)
    return revenue_setelah_diskon

# TODO: test
print(hitung_revenue(25000, 3))          # harusnya 75000
print(hitung_revenue(25000, 3, 0.1))     # harusnya 67500


75000
67500.0


---
## Task 5 — Terapkan Function ke DataFrame
Buat kolom `revenue` di df menggunakan `hitung_revenue()`.

Untuk mempermudah, kita bisa gunakan **vectorized** method : `df["revenue"] = hitung_revenue(df["price"], df["quantity"])`


In [18]:
# TODO: buat kolom baru "revenue" pakai hitung_revenue()
df["revenue"] = df.apply(lambda row: hitung_revenue(row['price'], row['quantity'], row.get('diskon', 0)), axis=1)

# TODO: tampilkan kolom order_id, customer, price, quantity, revenue (head)
df[["order_id", "customer", "price", "quantity", "revenue"]].head()

,order_id,customer,price,quantity,revenue
0,ORD-0001,Citra Pratama,176611,5,883055
1,ORD-0002,Indah Utami,93381,8,747048
2,ORD-0003,Eko Santoso,347289,2,694578
3,ORD-0004,Lestari Permana,63894,4,255576
4,ORD-0005,Kartika Lestari,71064,2,142128


---
## Task 6 — Function `count_orders_per_customer`
Function yang menerima DataFrame, mengembalikan **dict** `{customer: jumlah_order}`.

Petunjuk: `df["customer"].value_counts().to_dict()`


In [19]:
# TODO: definisikan function count_orders_per_customer(df) -> dict
def count_orders_per_customer(df):
    order_counts = df.groupby('customer')['order_id'].count()
    return order_counts.to_dict()

# TODO: panggil function, cek type-nya harus dict
order_counts = count_orders_per_customer(df)
print(type(order_counts))

# TODO: loop 5 pertama, cetak "nama: X order"
for i, (customer, count) in enumerate(list(order_counts.items())[:5]):
    print(f"{customer}: {count} order")


<class 'dict'>
Andi Saputra: 22 order
Budi Wijaya: 22 order
Citra Pratama: 14 order
Dewi Nugroho: 16 order
Eko Santoso: 9 order


---
## Task 7 — Siapa Customer dengan Total Belanja Terbesar?
Group by customer, jumlahkan revenue, ambil yang tertinggi.


In [25]:
# TODO: group by customer, jumlahkan revenue, sort descending
top_spender = df.groupby("customer")["revenue"].sum().sort_values(ascending=False)
# TODO: cetak top spender dan total belanjanya
print(top_spender.head())
print()
print(f"Top Spender: {top_spender.idxmax()}")
print(f"Total Spending: {top_spender.max()}")




customer
Indah Utami        120253746
Budi Wijaya         99314207
Hendra Setiawan     82082889
Kartika Lestari     68413023
Gita Rahman         60166111
Name: revenue, dtype: int64

Top Spender: Indah Utami
Total Spending: 120253746


---
## Bonus — Simpan Hasil Analisis ke CSV


In [26]:
# TODO: pilih kolom [order_id, customer, category, price, quantity, revenue, city]
df_filtered = df[["order_id", "customer", "category", "price", "quantity", "revenue", "city"]]
# TODO: simpan ke "hasil_analisis.csv" dengan index=False
df_filtered.to_csv("hasil_analisis.csv", index=False)
print("Data berhasil disimpan ke hasil_analisis.csv")

Data berhasil disimpan ke hasil_analisis.csv
